# Step 2d · EAGLE-I's own county customer totals (checking the denominator)

**Purpose**: the 2024 EAGLE-I file has one more column than the other years, `total_customers`, the number of customers in each county. This step extracts it and compares it with the denominator used in Step 2c, to answer two questions:

1. How far the customer totals I estimated as "county population × statewide customers per person" are from EAGLE-I's own figures
2. Whether `total_customers` is **all** customers of the county, or only the customers of the utilities **collected by EAGLE-I**. If the latter, it can be used to deal with the problem that "some counties are only partly covered"

**This step does not change the label table**; it only extracts and compares. Whether to revise the denominator on this basis is decided after seeing the results.

| Part | Content | Approx. time |
| --- | --- | --- |
| A | Download the 2024 file (about 1.4 GB) and summarize `total_customers` by county and month | 5–10 minutes |
| B | Compare with the Step 2c denominator and with each state's customer total | 1 minute |

No Earth Engine is needed.

In [ ]:
# 1. Environment, output folder, chart style
import os
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display

try:                                    # in Colab: mount Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    OUT, TMP = "/content/drive/MyDrive/us_flood_outage", "/content"
except ImportError:                     # in local Jupyter: use the current folder
    OUT = "us_flood_outage" if os.path.isdir("us_flood_outage") else "../us_flood_outage"    # next to the notebook, or one level up
    TMP = f"{OUT}/tmp"
os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)
print("Output folder:", OUT)

def tidy(df, n=1):
    # round numeric columns only, to keep tables readable
    return df.round({c: n for c in df.select_dtypes("number").columns})

STATE_ABBR = {"01": "AL", "12": "FL", "13": "GA", "22": "LA", "28": "MS", "37": "NC", "45": "SC", "48": "TX"}
C_POWER = "#2a78d6"
INK, MUTED, GRID, SURFACE, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb", "#b9b8b2"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

## A. Extract `total_customers`

Each row of the 2024 file is "customers out in one county in one 15-minute slot", and carries the county's `total_customers` along. Here its mean, minimum and maximum are summarized by "county × month". Looking month by month shows whether the figure changes during the year: a sudden jump in some month most likely means a utility was added to or removed from EAGLE-I.

The summary is saved to the output folder, so nothing is downloaded on a rerun.

In [ ]:
# 2. Download the 2024 file and summarize total_customers by "county × month"
FILE_ID = 53581661                                               # figshare id of the 2024 file
MONTHLY = f"{OUT}/step2d_total_customers_2024_monthly.csv"

def download(url, path, label="", min_bytes=1000):
    # download a file to path; skip it if it already exists
    if os.path.exists(path) and os.path.getsize(path) >= min_bytes:
        return path
    print(f"Downloading {label or url} …")
    with requests.get(url, stream=True, timeout=180, headers={"User-Agent": "Mozilla/5.0"}) as r:
        r.raise_for_status()
        done, shown = 0, 0
        with open(path + ".part", "wb") as f:
            for block in r.iter_content(chunk_size=1 << 22):
                f.write(block); done += len(block)
                if done - shown >= 400e6:
                    shown = done; print(f"  {done / 1e6:,.0f} MB downloaded")
    if done < min_bytes:
        os.remove(path + ".part")
        raise RuntimeError(f"The downloaded file is too small ({done} bytes); the site may have blocked the request. Open {url} in a browser, download it manually, "
                           f"place it in the us_flood_outage folder under the same name, then rerun this cell.")
    os.replace(path + ".part", path)
    return path

if not os.path.exists(MONTHLY):
    raw = f"{OUT}/eaglei_outages_2024.csv"                       # a manually downloaded copy can be placed here
    if not os.path.exists(raw):
        raw = download(f"https://ndownloader.figshare.com/files/{FILE_ID}", f"{TMP}/eaglei_outages_2024.csv", "EAGLE-I 2024", 300e6)
    header = {c.strip().lower(): c for c in pd.read_csv(raw, nrows=0, encoding="utf-8-sig").columns}
    if "total_customers" not in header:
        raise KeyError(f"This file has no total_customers column; the header is: {list(header.values())}")
    fips_col, time_col, tot_col = header["fips_code"], header["run_start_time"], header["total_customers"]
    parts = []
    for chunk in pd.read_csv(raw, usecols=[fips_col, time_col, tot_col], dtype={fips_col: str, time_col: str},
                             encoding="utf-8-sig", chunksize=2_000_000):
        chunk = chunk.assign(total=pd.to_numeric(chunk[tot_col], errors="coerce")).dropna(subset=["total", fips_col, time_col])
        key = [chunk[fips_col].str.split(".").str[0].str.zfill(5).rename("fips"), chunk[time_col].str[:7].rename("month")]
        parts.append(chunk.groupby(key)["total"].agg(["sum", "count", "min", "max"]))
    g = pd.concat(parts).groupby(["fips", "month"])
    monthly = pd.DataFrame({"mean": g["sum"].sum() / g["count"].sum(), "min": g["min"].min(), "max": g["max"].max(),
                            "records": g["count"].sum()}).reset_index()
    monthly.to_csv(MONTHLY, index=False)

monthly = pd.read_csv(MONTHLY, dtype={"fips": str, "month": str})
print(f"{monthly['fips'].nunique():,} counties, months {monthly['month'].min()} to {monthly['month'].max()}")

# one row per county: minimum and maximum over the year, and the mean in the months of Beryl (July) and Helene (September)
wide = monthly.pivot(index="fips", columns="month", values="mean")
county = pd.DataFrame({"year_min": monthly.groupby("fips")["min"].min(), "year_max": monthly.groupby("fips")["max"].max(),
                       "jul": wide.get("2024-07"), "sep": wide.get("2024-09"), "months": wide.notna().sum(axis=1)})
county["change_pct"] = (county["year_max"] / county["year_min"] - 1) * 100       # how far the maximum is above the minimum within the year
county.to_csv(f"{OUT}/step2d_total_customers_2024_by_county.csv")
ours = county[county.index.str[:2].isin(list(STATE_ABBR))]
print(f"{len(ours)} counties of the 8 states have a figure; in {int((ours['change_pct'] > 10).sum())} of them it changes by more than 10% during the year, in {int((ours['change_pct'] > 50).sum())} by more than 50%")
tidy(ours.sort_values("change_pct", ascending=False).head(10), 0)

## B. Comparison with the Step 2c denominator

Three tables and one figure:

1. **State totals**: the September `total_customers` of all counties are added up and compared with two 2022 figures from `coverage_history.csv`: the statewide customer total, and the customers collected (total × highest coverage). Whichever the sum is closer to tells which kind of figure `total_customers` is
2. **County by county**: for every county in the two 2024 events (Beryl and Helene), `total_customers ÷ customers_est` is computed. The table lists the median of this ratio by state, and the number of counties that are clearly low or high
3. **Counties whose denominator was raised**: for the counties where the peak exceeded the estimate in Step 2c, whether `total_customers` is large enough
4. **Figure**: my estimated customer total on the horizontal axis, EAGLE-I's `total_customers` on the vertical axis, one dot per county. Dots on the diagonal mean the two agree

In [ ]:
# 3. State totals, compared with coverage_history.csv
cov = pd.read_csv(f"{OUT}/step2c_coverage_by_state.csv")
cov = cov[cov["year"] == cov["year"].max()].set_index("state")
ours = county[county.index.str[:2].isin(list(STATE_ABBR))].assign(state=lambda d: d.index.str[:2].map(STATE_ABBR))
state_tab = pd.DataFrame({
    "counties_with_figure": ours.groupby("state").size(),
    "sum_total_customers_sep": ours.groupby("state")["sep"].sum(),
    "state_total_2022": cov["total_customers"],
    "collected_2022": cov["total_customers"] * cov["max_pct_covered"],
})
state_tab["vs_state_total_pct"] = state_tab["sum_total_customers_sep"] / state_tab["state_total_2022"] * 100
state_tab["vs_collected_pct"]   = state_tab["sum_total_customers_sep"] / state_tab["collected_2022"] * 100
state_tab.to_csv(f"{OUT}/step2d_state_comparison.csv")
tidy(state_tab, 0)

In [ ]:
# 4. County by county: the counties of the two 2024 events
labels = pd.read_csv(f"{OUT}/step2c_labels_county_event.csv", dtype={"fips": str})
cmp = labels[labels["event"].isin(["beryl2024", "helene2024"])].copy()
cmp["total_customers"] = np.where(cmp["event"] == "beryl2024", cmp["fips"].map(county["jul"]), cmp["fips"].map(county["sep"]))
cmp["ratio"] = cmp["total_customers"] / cmp["customers_est"]
cmp[["event", "fips", "name", "state", "customers_est", "customers_used", "peak_customers_out", "peak_pct", "hours_above", "hours_rel",
     "affected", "denom_raised", "total_customers", "ratio"]].to_csv(f"{OUT}/step2d_county_comparison.csv", index=False)

by_state = cmp.groupby(["event", "state"]).agg(
    counties=("fips", "size"), with_figure=("total_customers", "count"),
    sum_total_customers=("total_customers", "sum"), sum_estimate=("customers_est", "sum"),
    median_ratio=("ratio", "median"),
    below_0_5=("ratio", lambda r: int((r < 0.5).sum())), below_0_8=("ratio", lambda r: int((r < 0.8).sum())),
    above_1_25=("ratio", lambda r: int((r > 1.25).sum())), above_2=("ratio", lambda r: int((r > 2).sum())))
display(tidy(by_state, 2))

# counties whose denominator was raised: is total_customers at least as large as the cleaned peak
up = cmp[cmp["denom_raised"] & cmp["total_customers"].notna()]
print(f"Of the {len(up)} counties in the two 2024 events whose denominator was raised: total_customers is at least the peak in {int((up['total_customers'] >= up['peak_customers_out']).sum())};",
      f"the median of total_customers ÷ estimate is {up['ratio'].median():.2f}")
tidy(up.sort_values("ratio", ascending=False).head(12)[["event", "name", "customers_est", "peak_customers_out", "total_customers", "ratio"]], 2)

In [ ]:
# 5. Figure: my estimate against EAGLE-I's total_customers
d = cmp[cmp["total_customers"].notna() & (cmp["customers_est"] > 0)]
lo, hi = float(min(d["customers_est"].min(), d["total_customers"].min())) * 0.7, float(max(d["customers_est"].max(), d["total_customers"].max())) * 1.4
fig, ax = plt.subplots(figsize=(6.4, 6))
ax.plot([lo, hi], [lo, hi], color=FAINT, lw=1)
ax.scatter(d["customers_est"], d["total_customers"], s=12, color=C_POWER, edgecolor=SURFACE, linewidth=0.4, alpha=0.65)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect("equal")
ax.set_xlabel("Customers estimated from population (Step 2c)")
ax.set_ylabel("total_customers in the 2024 EAGLE-I file")
ax.set_title("Two county customer totals compared", pad=22)
within = ((d["ratio"] > 0.8) & (d["ratio"] < 1.25)).mean() * 100
ax.text(0, 1.03, f"Counties of Beryl and Helene (n = {len(d)}); {within:.0f}% within 0.8–1.25 of the estimate",
        transform=ax.transAxes, color=MUTED, fontsize=9)
fig.savefig(f"{OUT}/step2d_fig_customer_totals.png"); plt.show()
print(f"{within:.0f}% of the counties have a ratio between 0.8 and 1.25; the median ratio is {d['ratio'].median():.2f}")

## Output files

All written to the `us_flood_outage` folder:

- `step2d_total_customers_2024_monthly.csv`, `step2d_total_customers_2024_by_county.csv`
- `step2d_state_comparison.csv`, `step2d_county_comparison.csv`
- `step2d_fig_customer_totals.png`

Interpretation:

- In the table of cell 3, if `vs_state_total_pct` is close to 100, `total_customers` is all customers; if `vs_collected_pct` is close to 100 while the former is clearly lower (the difference is clearest in Mississippi), it only includes the customers collected
- In the table of cell 4, a `median_ratio` close to 1 with few counties in `below_0_5` and `above_2` means the population-based denominator is acceptable